In [ ]:
import pandas as pd

from load import LoadDataCSV

import torch

# Load datasets

df_loader=LoadDataCSV()

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

# Load text fields for training
post_texts = df_posts['text'].tolist()
fact_check_texts = df_fact_checks['claim'].apply(lambda x: x[0] if isinstance(x, tuple) else x).tolist()


In [3]:
from sentence_transformers import SentenceTransformer, util

# Load bi-encoder model and tokenizer
model = SentenceTransformer('all-mpnet-base-v2')  # Adjust as needed

# Generate embeddings for fact-check claims
fact_check_embeddings = model.encode(fact_check_texts, convert_to_tensor=True)


In [ ]:

# Load bi-encoder model and tokenizer
model = SentenceTransformer('all-mpnet-base-v2')  # Adjust as needed

# Generate embeddings for fact-check claims
fact_check_embeddings = model.encode(fact_check_texts, convert_to_tensor=True)


In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, losses

# Prepare training examples
train_examples = []
for _, row in df_fact_check_post_mapping.iterrows():
    post_id, fact_check_id = row['post_id'], row['fact_check_id']
    post_text = df_posts.loc[post_id, 'text']
    fact_check_text = df_fact_checks.loc[fact_check_id, 'claim'][0]
    train_examples.append(InputExample(texts=[post_text, fact_check_text], label=1.0))

# Set up DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

# Fine-tune the model
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=1, warmup_steps=100)


In [ ]:
# Embed all posts
post_embeddings = model.encode(post_texts, convert_to_tensor=True)

# Retrieve top-k fact-checks using cosine similarity
k = 10
top_k_results = {}
for idx, post_embedding in enumerate(post_embeddings):
    # Calculate cosine similarities between the post and all fact-check embeddings
    cos_scores = util.cos_sim(post_embedding, fact_check_embeddings)[0]
    top_k = torch.topk(cos_scores, k=k)
    
    # Retrieve fact_check IDs
    post_id = df_posts.index[idx]
    top_k_fact_checks = [df_fact_checks.index[i] for i in top_k.indices]
    top_k_results[post_id] = top_k_fact_checks


In [ ]:
import json

# Convert to JSON format
submission_data = [{"post_id": post_id, "retrieved_fact_checks": top_k_results[post_id]} for post_id in top_k_results]
with open('submission.json', 'w') as f:
    json.dump(submission_data, f, indent=4)
